# BIOMASS tomogram processor

This notebook opens a small real TOM L2A subset, derives power and decibels, extracts horizontal and vertical slices, calculates a vertical profile, and wraps the result for the shared data interface. It also records the important distinction between the 2 m vertical sampling in this product and an assumed nominal 20 m physical vertical resolution.

Install the optional dependencies with `python -m pip install -e '.[tomogram,examples]'`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from tomo_canopy.processors import TomogramProcessor
from tomo_canopy.transforms import match_vertical_resolution

## Locate the real L2A product

The provider metadata are inspected first. This does not open or load the multi-gigabyte measurement cube.

In [ ]:
project_root = next(
    path
    for path in (Path.cwd(), Path.cwd().parent)
    if (path / "pyproject.toml").exists()
)
tom_directory = project_root / "data" / "TOM"

processor = TomogramProcessor()
products = [
    processor.provider.inspect(path)
    for path in sorted(tom_directory.iterdir())
]
l2a_product = next(
    product for product in products if product.product_level == "L2A"
)
l2a_product

## Open a lazy geographic subset

The subset is centred on the product footprint and spans approximately 0.04° in each horizontal direction. Only the HV polarization and heights from −20 m to 60 m are retained. Passing `chunks="auto"` keeps the measurement data Dask-backed until a plot or calculation requests values.

In [ ]:
# Derive the centre from normalized coordinates. Some prototype products
# store their raw STAC bbox latitude-first, so product.bbox is not used here.
footprint = processor.open(
    l2a_product.root, polarization="HV", height_range=(0.0, 0.0),
    chunks="auto",
)
centre_x = float(footprint.x.mean())
centre_y = float(footprint.y.mean())
footprint.close()

tomogram = processor.open(
    l2a_product.root,
    expected_level="L2A",
    polarization="HV",
    height_range=(-20.0, 60.0),
    bounds=(
        centre_x - 0.02, centre_y - 0.02,
        centre_x + 0.02, centre_y + 0.02,
    ),
    chunks="auto",
)
processor.validate(tomogram, require_geocoded=True)
tomogram

## Sampling is not physical resolution

The coordinate spacing is read from the product. The nominal physical resolution must come from authoritative product documentation or processing metadata. Here it is explicitly set to 20 m for the example. Adjacent 2 m samples are therefore not independent 2 m vertical observations.

In [ ]:
vertical_sampling_m = float(tomogram.z.diff("z").median())
NOMINAL_VERTICAL_RESOLUTION_M = 20.0  # Verify for the selected product.

print(f"Vertical sampling: {vertical_sampling_m:g} m")
print(
    f"Assumed physical vertical resolution: "
    f"{NOMINAL_VERTICAL_RESOLUTION_M:g} m"
)

## Derive linear power and decibels

The native cube is preserved. Derived variables are added lazily, so provenance remains visible and the pipeline stays composable.

In [ ]:
power = processor.to_power(tomogram)
power_db = processor.to_db(power, floor_db=-40.0)
power_db[["tomographic_power", "tomographic_power_db"]]

## Horizontal slices

These maps show reconstructed power at selected relative heights. Fine changes between adjacent height samples should be interpreted within the broader physical vertical-resolution limit.

In [ ]:
requested_heights = [0.0, 20.0, 40.0]
horizontal = power_db.tomographic_power_db.squeeze("polarization").sel(
    z=requested_heights, method="nearest"
)
horizontal.plot(
    x="x", y="y", col="z", col_wrap=3,
    cmap="viridis", robust=True,
)
plt.suptitle("BIOMASS L2A HV tomographic power", y=1.04)
plt.show()

## Vertical slice

A vertical plane is selected at the longitude nearest the centre of the subset. It shows broad vertical scattering zones, not independently resolved 2 m canopy layers.

In [ ]:
vertical = processor.vertical_slice(
    power_db, axis="x", coordinate=centre_x
)
vertical.tomographic_power_db.squeeze("polarization").plot(
    x="y", y="z", cmap="viridis", robust=True, figsize=(10, 5)
)
plt.title(
    f"HV vertical slice at x={vertical.attrs['selected_slice_coordinate']:.4f}"
)
plt.show()

## Mean vertical profile

The profile averages the selected horizontal footprint. Peaks identify broad scattering zones, but their apparent width and separation remain constrained by the physical vertical resolution.

In [ ]:
profile = processor.vertical_profile(power_db, statistic="mean")
profile.tomographic_power_db.squeeze("polarization").plot(
    y="z", marker="o", markersize=3, figsize=(5, 7)
)
plt.title("Mean HV vertical power profile")
plt.xlabel("Power [dB]")
plt.ylabel("Relative tomographic height [m]")
plt.grid(alpha=0.3)
plt.show()

## Record the effective resolution

Because this tomogram is already assumed to have 20 m physical resolution, `source_resolution` and `target_resolution` are both set to 20 m. The transform records sampling and resolution metadata without blurring the tomogram a second time.

For comparison, a higher-resolution TLS-derived volume would use its own source resolution and the same 20 m target. That operation belongs in `tomo_canopy.transforms`, not in either sensor processor.

In [ ]:
resolution_aware = match_vertical_resolution(
    power,
    source_resolution=NOMINAL_VERTICAL_RESOLUTION_M,
    target_resolution=NOMINAL_VERTICAL_RESOLUTION_M,
    variables=["tomographic_power"],
)
{
    key: resolution_aware.attrs[key]
    for key in ("vertical_sampling", "vertical_resolution")
}

## Convert to the shared data interface

A geocoded L2A dataset can become a regular volume layer. Native L1D remains in SAR geometry and is intentionally rejected here until a geocoding transform is available.

In [ ]:
tomogram_layer = processor.to_data_layer(
    resolution_aware, name="biomass_tomogram_hv"
)
tomogram_layer